# M3L2 E05 - RAG chat con memoria

## Objetivo

Combinamos RAG y memoria para responder preguntas de seguimiento sobre documentos.

## Que aporta cada parte

| Parte | Aporta |
|---|---|
| Retriever | Informacion del documento |
| Historial | Lo que se dijo antes |
| Prompt | Instrucciones y formato |
| Modelo | Redaccion de la respuesta |

## Diagrama

```text
Pregunta -> Retriever -> contexto documental
Pregunta -> historial por session_id
contexto + historial -> prompt -> modelo -> respuesta
```

In [ ]:
# !pip install langchain langchain-openai langchain-community chromadb

In [ ]:
import os
import getpass
from langchain_openai import ChatOpenAI

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Ingresa tu OpenAI API key: ")


def obtener_modelo(temperature: float = 0.2):
    return ChatOpenAI(model="gpt-4o-mini", temperature=temperature)


print("API key cargada en la variable de entorno OPENAI_API_KEY")

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

llm = obtener_modelo()
historiales = {}

In [ ]:
Path("omia.txt").write_text(
    "OMIA es una metodologia interna para organizar proyectos de AI Engineering.\n"
    "Sus etapas son observar, modularizar, implementar y auditar.\n"
    "Observar significa entender el problema antes de escribir codigo.\n"
    "Modularizar significa separar responsabilidades en componentes claros.\n"
    "Auditar significa revisar trazabilidad, errores y calidad antes de entregar.\n",
    encoding="utf-8",
)

In [ ]:
loader = TextLoader("omia.txt", encoding="utf-8")
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=140, chunk_overlap=20)
chunks = splitter.split_documents(docs)
embeddings = OpenAIEmbeddings()
vectorstore = Chroma.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print("Chunks:", len(chunks))

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Responde usando el contexto documental y el historial. Si no alcanza, dilo con claridad."),
    MessagesPlaceholder("historial"),
    ("human", "Contexto documental:\n{contexto}\n\nPregunta actual: {pregunta}"),
])
cadena_base = prompt | llm | StrOutputParser()

In [ ]:
def obtener_historial(session_id: str):
    if session_id not in historiales:
        historiales[session_id] = InMemoryChatMessageHistory()
    return historiales[session_id]

chat_rag = RunnableWithMessageHistory(
    cadena_base,
    obtener_historial,
    input_messages_key="pregunta",
    history_messages_key="historial",
)

In [ ]:
def preguntar(pregunta: str, session_id: str = "demo"):
    docs_recuperados = retriever.invoke(pregunta)
    contexto = format_docs(docs_recuperados)
    return chat_rag.invoke(
        {"pregunta": pregunta, "contexto": contexto},
        config={"configurable": {"session_id": session_id}},
    )

print(preguntar("Que significa modularizar?"))
print(preguntar("Y que dije recien?"))

## Errores comunes

| Error | Causa |
|---|---|
| Contesta sin documentos | No se paso `contexto` |
| No recuerda | No se uso `RunnableWithMessageHistory` |
| Mezcla usuarios | No se separo por `session_id` |

In [ ]:
assert retriever is not None
assert chat_rag is not None
assert "demo" in historiales
print("Checks OK")

## Resumen

RAG responde con documentos. Memoria mantiene la conversacion.